In [ ]:
import os
import cv2
import json
import pickle
import tempfile
import numpy as np
import torch
import zipfile
import time
import urllib.request
from pathlib import Path
from flask import Flask, request, jsonify
from flask_cors import CORS

# ── MediaPipe (Tasks API) ─────────────────────────────────────────────────
try:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_tasks
    from mediapipe.tasks.python import vision as mp_vision
    MP_AVAILABLE = True

    _MODEL_FILE = "pose_landmarker_lite.task"
    _MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
    if not os.path.exists(_MODEL_FILE):
        print("📥 Downloading MediaPipe pose model...")
        urllib.request.urlretrieve(_MODEL_URL, _MODEL_FILE)
        print(f"✅ Downloaded {_MODEL_FILE}")

except ImportError:
    MP_AVAILABLE = False
    print("⚠️  mediapipe not installed — using random keypoints for testing")

app = Flask(__name__)
CORS(app)

# ═══════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════════════════
MODEL_PATH  = os.environ.get("AE_MODEL_PATH",  "autoencoder_prod.pt")
SCALER_PATH = os.environ.get("AE_SCALER_PATH", "scaler.pkl")

SEQ_LEN       = 50          # frames the model expects
N_RAW_KP      = 24          # raw keypoints (x,y × 12 joints)
N_ENG_FEAT    = 48          # engineered features (output dim)
ANOMALY_THRESH = 0.15       # MAE threshold above which a frame is anomalous

# MediaPipe joint indices we use (12 joints × x,y = 24 values)
# Indices from MediaPipe BlazePose 33-landmark model
JOINT_INDICES = [
    11, 12,   # shoulders L/R
    13, 14,   # elbows L/R
    15, 16,   # wrists L/R
    23, 24,   # hips L/R
    25, 26,   # knees L/R
    27, 28,   # ankles L/R
]

# Joint group definitions for anomaly reporting
# Maps to the 24 raw features (2 values each joint: x at i*2, y at i*2+1)
JOINT_GROUPS = {
    "shoulder_symmetry":           [0, 1, 2, 3],       # L/R shoulder x,y
    "left_shoulder":               [0, 1],
    "right_shoulder":              [2, 3],
    "left_elbow":                  [4, 5],
    "right_elbow":                 [6, 7],
    "left_wrist":                  [8, 9],
    "right_wrist":                 [10, 11],
    "hip_symmetry":                [12, 13, 14, 15],    # L/R hip x,y
    "left_hip":                    [12, 13],
    "right_hip":                   [14, 15],
    "left_knee":                   [16, 17],
    "right_knee":                  [18, 19],
    "left_ankle":                  [20, 21],
    "right_ankle":                 [22, 23],
    "contralateral_leftSH_rightHP": [0, 1, 14, 15],    # L shoulder + R hip
    "contralateral_rightSH_leftHP": [2, 3, 12, 13],    # R shoulder + L hip
    "torso_sway":                  [0, 2, 12, 14],      # shoulder/hip x coords
    "trunk_inclination":           [0, 2, 12, 14, 1, 3, 13, 15],
    "shoulder_hip_coupling_right": [2, 3, 14, 15],
    "shoulder_hip_coupling_left":  [0, 1, 12, 13],
}

CLINICAL_NOTES = {
    "shoulder_symmetry":           "Reduced left-right gait symmetry across shoulder girdle coordinates.",
    "left_shoulder":               "Localized anomaly near left shoulder joint plane.",
    "right_shoulder":              "Localized anomaly near right shoulder joint plane.",
    "left_elbow":                  "Irregular arm swing pattern near left elbow.",
    "right_elbow":                 "Irregular arm swing pattern near right elbow.",
    "left_wrist":                  "Atypical wrist movement during left arm swing.",
    "right_wrist":                 "Atypical wrist movement during right arm swing.",
    "hip_symmetry":                "Asymmetric hip movement detected — possible antalgic gait.",
    "left_hip":                    "Deviation in left hip trajectory. Check for Trendelenburg sign.",
    "right_hip":                   "Deviation in right hip trajectory. Check for Trendelenburg sign.",
    "left_knee":                   "Irregular flexion/extension trace near left knee indicators.",
    "right_knee":                  "Irregular extension/flexion trace mapped to right knee tracking.",
    "left_ankle":                  "Atypical gait strike pattern near left ankle coordinate vectors.",
    "right_ankle":                 "Atypical gait strike pattern or ground impact near right ankle.",
    "contralateral_leftSH_rightHP":"Abnormal contralateral arm-leg coordination (Left Shoulder / Right Hip).",
    "contralateral_rightSH_leftHP":"Abnormal contralateral arm-leg coordination (Right Shoulder / Left Hip).",
    "torso_sway":                  "Excessive lateral trunk sway — potential balance impairment.",
    "trunk_inclination":           "Abnormal trunk inclination angle during gait cycle.",
    "shoulder_hip_coupling_right": "Disrupted right-side shoulder-hip coupling rhythm.",
    "shoulder_hip_coupling_left":  "Disrupted left-side shoulder-hip coupling rhythm.",
}

# ═══════════════════════════════════════════════════════════════════════════
#  MODEL LOADING
# ═══════════════════════════════════════════════════════════════════════════
ae_model = None
scaler    = None

def _load_model():
    global ae_model, scaler

    # ── Load autoencoder ──────────────────────────────────────────────────
    # If the path is a directory (extracted zip), repack it on the fly
    mp_path = Path(MODEL_PATH)

    if mp_path.is_dir():
        print("📦 Model is a directory — repacking as .pt …")
        tmp_pt = "/tmp/_ae_repacked.pt"
        now = time.time()
        for root, dirs, files in os.walk(mp_path):
            for f in files:
                os.utime(os.path.join(root, f), (now, now))
        with zipfile.ZipFile(tmp_pt, "w", zipfile.ZIP_STORED) as zf:
            for root, dirs, files in os.walk(mp_path):
                for file in sorted(files):
                    fp = Path(root) / file
                    arcname = fp.relative_to(mp_path.parent)
                    zf.write(fp, arcname)
        load_path = tmp_pt
    else:
        load_path = str(mp_path)

    ae_model = torch.jit.load(load_path, map_location="cpu")
    ae_model.eval()
    print(f"✅ Autoencoder loaded  →  input (1, N, {N_RAW_KP})  output (1, 50, {N_ENG_FEAT})")

    # ── Load scaler ───────────────────────────────────────────────────────
    if os.path.exists(SCALER_PATH):
        with open(SCALER_PATH, "rb") as f:
            scaler = pickle.load(f)
        print(f"✅ Scaler loaded from {SCALER_PATH}")
    else:
        print(f"⚠️  scaler.pkl not found — running WITHOUT scaling (results may be less accurate)")


# ═══════════════════════════════════════════════════════════════════════════
#  KEYPOINT EXTRACTION  (MediaPipe Tasks API — VIDEO mode)
# ═══════════════════════════════════════════════════════════════════════════
def extract_keypoints_from_video(video_path: str) -> np.ndarray:
    """
    Extract keypoints from video using MediaPipe Tasks PoseLandmarker (VIDEO mode).

    Returns:
        np.ndarray shape (N_frames, 24) — raw x,y for 12 joints, normalized 0-1
    """
    if not MP_AVAILABLE:
        print("⚠️  Using random keypoints (mediapipe not available)")
        return np.random.randn(SEQ_LEN, N_RAW_KP).astype(np.float32)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS) or 30.0
    print(f"📹 Video: {total_frames} frames @ {fps:.1f} fps")

    keypoints_list = []

    base_opts = mp_tasks.BaseOptions(model_asset_path=_MODEL_FILE)
    options = mp_vision.PoseLandmarkerOptions(
        base_options=base_opts,
        running_mode=mp_vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        frame_idx = 0
        last_ts = -1
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Process every other frame for speed on long videos
            if total_frames > 200 and frame_idx % 2 != 0:
                frame_idx += 1
                continue

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # timestamps must be strictly increasing for VIDEO mode
            timestamp_ms = int((frame_idx / fps) * 1000)
            if timestamp_ms <= last_ts:
                timestamp_ms = last_ts + 1
            last_ts = timestamp_ms

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                lm  = result.pose_landmarks[0]   # first person
                row = []
                for idx in JOINT_INDICES:
                    if idx < len(lm):
                        row.append(lm[idx].x)
                        row.append(lm[idx].y)
                    else:
                        row.extend([0.0, 0.0])
                keypoints_list.append(row)

            frame_idx += 1

    cap.release()

    if len(keypoints_list) < 10:
        raise ValueError(
            f"Too few valid frames detected ({len(keypoints_list)}). "
            "Ensure the person is clearly visible and well-lit."
        )

    kp_array = np.array(keypoints_list, dtype=np.float32)
    print(f"✅ Extracted {kp_array.shape[0]} valid frames → shape {kp_array.shape}")
    return kp_array


# ═══════════════════════════════════════════════════════════════════════════
#  WINDOWING  — sliding window to get SEQ_LEN frames
# ═══════════════════════════════════════════════════════════════════════════
def get_windows(kp: np.ndarray, window_size: int = SEQ_LEN) -> np.ndarray:
    """
    If video has ≥ window_size frames: use sliding windows (step=1).
    If video has < window_size frames: pad with last frame.

    Returns:
        np.ndarray shape (n_windows, window_size, N_RAW_KP)
    """
    n = kp.shape[0]

    if n < window_size:
        # Pad
        pad = np.tile(kp[-1:], (window_size - n, 1))
        kp  = np.vstack([kp, pad])
        return kp[np.newaxis]  # (1, window_size, 24)

    # Sliding windows
    windows = np.stack([kp[i:i+window_size] for i in range(0, n - window_size + 1, 5)])
    return windows  # (n_windows, window_size, 24)


# ═══════════════════════════════════════════════════════════════════════════
#  LOCALIZATION CORE
# ═══════════════════════════════════════════════════════════════════════════
def localize_anomalies(kp_raw: np.ndarray) -> dict:
    """
    Run LSTM Autoencoder → compute reconstruction errors → localize joints.

    Args:
        kp_raw: (N_frames, 24) raw normalized keypoints

    Returns:
        dict with full localization report
    """
    # 1. Scale
    if scaler is not None:
        n, f = kp_raw.shape
        kp_scaled = scaler.transform(kp_raw.reshape(-1, f)).reshape(n, f)
    else:
        kp_scaled = kp_raw.copy()

    # 2. Window
    windows = get_windows(kp_scaled, SEQ_LEN)  # (W, 50, 24)
    n_windows = windows.shape[0]

    # 3. Run model on all windows, collect errors
    all_frame_errors   = []   # per-frame MAE across all features
    all_feature_errors = []   # per-feature MAE across all frames

    with torch.no_grad():
        for i in range(n_windows):
            x = torch.FloatTensor(windows[i]).unsqueeze(0)  # (1, 50, 24)
            reconstruction = ae_model(x)                     # (1, 50, 48)

            # Reconstruction is in 48-feature space
            # We compare the 24 raw inputs with the first 24 dims of reconstruction
            # (the model outputs 48 features — raw 24 + derived 24)
            recon_24 = reconstruction[0, :, :N_RAW_KP].numpy()   # (50, 24)
            orig_24  = windows[i]                                  # (50, 24)

            frame_mae   = np.mean(np.abs(orig_24 - recon_24), axis=1)   # (50,)
            feature_mae = np.mean(np.abs(orig_24 - recon_24), axis=0)   # (24,)

            all_frame_errors.append(frame_mae)
            all_feature_errors.append(feature_mae)

    # 4. Aggregate across windows
    all_frame_errors   = np.stack(all_frame_errors)    # (W, 50)
    all_feature_errors = np.stack(all_feature_errors)  # (W, 24)

    mean_frame_errors   = np.mean(all_frame_errors,   axis=0)  # (50,) temporal
    mean_feature_errors = np.mean(all_feature_errors, axis=0)  # (24,) spatial

    # 5. Temporal: find peak anomalous frame
    peak_frame_idx = int(np.argmax(mean_frame_errors))
    peak_frame_val = float(mean_frame_errors[peak_frame_idx])

    # 6. Spatial: score each joint group
    joint_scores = {}
    for group_name, feat_indices in JOINT_GROUPS.items():
        valid_idx = [i for i in feat_indices if i < N_RAW_KP]
        if not valid_idx:
            continue
        group_score = float(np.mean(mean_feature_errors[valid_idx]))
        joint_scores[group_name] = group_score

    # 7. Rank joints
    ranked = sorted(joint_scores.items(), key=lambda x: x[1], reverse=True)

    # 8. Top 5
    top5 = []
    total_score = sum(s for _, s in ranked)
    for name, score in ranked[:5]:
        contrib = (score / total_score * 100) if total_score > 0 else 0
        top5.append({
            "name":         name,
            "score":        round(score, 5),
            "contribution": round(contrib, 1),
            "clinical":     CLINICAL_NOTES.get(name, "Anomaly detected in this joint group."),
        })

    # 9. Highest variance raw feature
    highest_var_idx  = int(np.argmax(mean_feature_errors))
    joint_idx        = highest_var_idx // 2
    coord            = "x" if highest_var_idx % 2 == 0 else "y"
    joint_labels     = [
        "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
        "left_wrist",    "right_wrist",    "left_hip",   "right_hip",
        "left_knee",     "right_knee",     "left_ankle", "right_ankle",
    ]
    joint_label = joint_labels[joint_idx] if joint_idx < len(joint_labels) else f"joint_{joint_idx}"
    highest_var_col = f"{joint_label}_{coord}"

    # 10. Frame timeline for UI (downsampled to 50 points)
    timeline = mean_frame_errors.tolist()

    return {
        # Core localization
        "peak_frame":            peak_frame_idx,
        "peak_frame_error":      round(peak_frame_val, 5),
        "primary_joint":         ranked[0][0] if ranked else "unknown",
        "highest_variance_col":  highest_var_col,
        "frames_analyzed":       SEQ_LEN,
        "windows_processed":     n_windows,

        # Top 5 joints
        "top_joints": top5,

        # Full timeline (for chart)
        "frame_errors_timeline": [round(v, 5) for v in timeline],

        # All joint scores (for full bar chart)
        "all_joint_scores": {k: round(v, 5) for k, v in ranked},

        # Status
        "anomaly_detected": bool(peak_frame_val > ANOMALY_THRESH),
        "overall_error":    round(float(np.mean(mean_frame_errors)), 5),
    }


# ═══════════════════════════════════════════════════════════════════════════
#  FLASK ROUTES
# ═══════════════════════════════════════════════════════════════════════════
@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status":       "ok",
        "model_loaded": ae_model is not None,
        "scaler_loaded": scaler is not None,
        "input_shape":  f"(1, {SEQ_LEN}, {N_RAW_KP})",
        "output_shape": f"(1, {SEQ_LEN}, {N_ENG_FEAT})",
    })


@app.route("/localize_gait", methods=["POST"])
def localize_gait():
    # ── Validate ──────────────────────────────────────────────────────────
    if "video" not in request.files:
        return jsonify({"error": "No video file — send as multipart/form-data with key 'video'"}), 400

    if ae_model is None:
        return jsonify({"error": "Model not loaded — check server logs"}), 503

    video_file = request.files["video"]
    if video_file.filename == "":
        return jsonify({"error": "Empty filename"}), 400

    # ── Save temp ─────────────────────────────────────────────────────────
    suffix = Path(video_file.filename).suffix or ".mp4"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        video_file.save(tmp.name)
        tmp_path = tmp.name

    try:
        # ── Extract keypoints ─────────────────────────────────────────────
        kp_raw = extract_keypoints_from_video(tmp_path)

        # ── Localize ──────────────────────────────────────────────────────
        report = localize_anomalies(kp_raw)

        return jsonify(report), 200

    except ValueError as e:
        return jsonify({"error": str(e)}), 422
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": f"Inference error: {str(e)}"}), 500
    finally:
        os.unlink(tmp_path)


# ═══════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    print("=" * 60)
    print("  GaitAI Localization Server — Model 02")
    print("=" * 60)
    _load_model()
    app.run(host="0.0.0.0", port=5001, debug=False)

  GaitAI Localization Server — Model 02
✅ Autoencoder loaded  →  input (1, N, 24)  output (1, 50, 48)
⚠️  scaler.pkl not found — running WITHOUT scaling (results may be less accurate)
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.20.10.3:5001
Press CTRL+C to quit


📹 Video: 1543 frames @ 28.8 fps


e:\gaitai\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✅ Extracted 772 valid frames → shape (772, 24)


172.20.10.2 - - [16/Jun/2026 12:43:12] "POST /localize_gait HTTP/1.1" 200 -
